In [1]:
%pip install datasets math_verify vllm torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [3]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

data/train-00000-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00001-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

data/train-00002-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/train-00007-of-00010.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00008-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00009-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93733 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [8]:
import gc
import torch
from vllm import LLM, SamplingParams

for var in ["llm", "generations"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    gpu_memory_utilization=0.6,
)
sampling_params = SamplingParams(
    n=8,
    max_tokens=512,
)

prompts = ds["train"]["problem"][:1]
generations = llm.generate(prompts, sampling_params)
for prompt, generation in zip(prompts, generations):
    print("*"*50 + " Prompt " + "*"*50)
    print(prompt)
    for i, output in enumerate(generation.outputs):
        print("*"*50 + f" Generation {i+1} ", "*"*50)
        print(output.text)

INFO 08-23 21:01:37 [utils.py:612] [shutdown] Process manager: send sigterm to process EngineCore
(EngineCore pid=825) INFO 08-23 21:01:37 [core.py:1332] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=825) INFO 08-23 21:01:37 [core.py:1468] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=825) INFO 08-23 21:01:37 [core.py:1499] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=825) INFO 08-23 21:01:37 [core.py:1345] [shutdown] EngineCore: exiting busy loop
INFO 08-23 21:01:41 [api_utils.py:273] non-default args: {'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 08-23 21:01:41 [model.py:645] Resolved architecture: Qwen2ForCausalLM
WARNING 08-23 21:01:41 [model.py:2164] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 08-23 21:01:41 [model.py:2

[W823 21:01:59.857357061 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=953) INFO 08-23 21:02:00 [model_runner.py:308] Loading model from scratch...
(EngineCore pid=953) ERROR 08-23 21:02:01 [fa_utils.py:273] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=953) INFO 08-23 21:02:02 [cuda.py:482] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=953) INFO 08-23 21:02:02 [weight_utils.py:867] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 3.31 GiB. Available RAM: 8.90 GiB.
(EngineCore pid=953) INFO 08-23 21:02:02 [weight_utils.py:890] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.45s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.45s/it]
(EngineCore pid=953) 


(EngineCore pid=953) INFO 08-23 21:02:05 [default_loader.py:430] Loading weights took 2.55 seconds
(EngineCore pid=953) INFO 08-23 21:02:06 [model_runner.py:329] Model loading took 3.45 GiB and 5.649481 seconds
(EngineCore pid=953) WARNING 08-23 21:02:06 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=953) INFO 08-23 21:02:08 [caching.py:335] reconstructed serializable fn from standalone compile artifacts. num_artifacts=3 num_submods=29
(EngineCore pid=953) INFO 08-23 21:02:08 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/af68ece8784a1aa24e6a420cecb0d4c710dcfcb24120464d45f2e760c657287a/rank_0_0/model
(EngineCore pid=953) INFO 08-23 21:02:08 [monitor.py:53] torch.compile took 0.20 s in total
(EngineCore pid=953) INFO 08-23 21:02:08 [monitor.py:81] Initial profiling/warmup run took 0.09

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=953) WARNING 08-23 21:02:20 [jit_monitor.py:135] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 8/8 [00:09<00:00,  1.15s/it, est. speed input: 114.00 toks/s, output: 445.55 toks/s]

************************************************** Prompt **************************************************
## Task B-1.3.

A ship traveling along a river has covered $24 \mathrm{~km}$ upstream and $28 \mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \mathrm{~km}$ upstream and $21 \mathrm{~km}$ downstream, or half an hour more than for traveling $15 \mathrm{~km}$ upstream and $42 \mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.

Determine the speed of the ship in still water and the speed of the river.
************************************************** Generation 1  **************************************************
 
Note: Assume that both the ship and the river move uniformly.

First, they tell us to let u denote the speed of the ship in still water and v denote the speed of the river.

So we are to set up equations based on these variables.

Let me try to process the problem step by step.

First, let's l